# TRM + 1-Step Gradient (1st Cell of 2x2 Matrix)
### Paper 1: Disentangling Gradient Quality from Architecture

hidden=384, L_layers=2, H_cycles=3, L_cycles=6, 500+500 examples, 15000 epochs
~8.5h on 2xT4

## 1. Setup

In [1]:
!pip install -q einops coolname argdantic huggingface_hub hydra-core omegaconf pydantic wandb tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.9 MB/s eta 0:00:00


In [2]:
import os, sys, yaml, json
import torch
import numpy as np
from pathlib import Path
print(f'PyTorch: {torch.__version__}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
GPU_COUNT = torch.cuda.device_count()

PyTorch: 2.10.0+cu128
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
REPO_DIR = '/kaggle/working/TinyRecursiveModels'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SamsungSAILMontreal/TinyRecursiveModels.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

Cloning into '/kaggle/working/TinyRecursiveModels'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 81 (delta 1), reused 0 (delta 0), pack-reused 79 (from 2)
Receiving objects: 100% (81/81), 1.25 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Working dir: /kaggle/working/TinyRecursiveModels


## 2. Build Dataset

In [4]:
!python dataset/build_sudoku_dataset.py --subsample-size 500 --num-aug 500 --output-dir data/sudoku-extreme-1k

train.csv: 100%|██████████████████████████████| 719M/719M [00:04<00:00, 177MB/s]
100%|█████████████████████████████████████████| 500/500 [00:27<00:00, 18.18it/s]
test.csv: 100%|████████████████████████████| 79.4M/79.4M [00:01<00:00, 49.2MB/s]
100%|██████████████████████████████| 422786/422786 [00:00<00:00, 1825111.58it/s]


## 3. Patches (AdamW + periodic print)

In [5]:
# Patch pretrain.py: AdamW swap + periodic print + limit eval to 2K puzzles
!git -C {REPO_DIR} checkout -- pretrain.py
with open(f"{REPO_DIR}/pretrain.py", "r") as f:
    src = f.read()
# 1. AdamW swap
src = src.replace("from adam_atan2 import AdamATan2", "from torch.optim import AdamW as AdamATan2  # T4 compat")
# 2. Periodic print every 200 steps
old = "                wandb.log(metrics, step=train_state.step)"
new = ("                if train_state.step % 200 == 0:\n"
       '                    print(f"[Step {train_state.step}/{train_state.total_steps}] lm_loss={metrics.get(\'train/lm_loss\',0):.2f} acc={metrics.get(\'train/accuracy\',0):.3f} exact={metrics.get(\'train/exact_accuracy\',0):.4f} steps={metrics.get(\'train/steps\',0):.1f}", flush=True)\n'
       "                wandb.log(metrics, step=train_state.step)")
src = src.replace(old, new)
# 3. Limit eval to 2048 test puzzles (8 batches x 256) — saves 1-2h
src = src.replace("processed_batches += 1", "processed_batches += 1\n            if processed_batches >= 8:\n                break")
with open(f"{REPO_DIR}/pretrain.py", "w") as f:
    f.write(src)
print("Patched: AdamW + periodic print + eval limited to ~2K puzzles.")


Patched: AdamW + periodic print + eval limited to ~2K puzzles.


In [6]:
# Patch trm.py: 1-step gradient (O(1) memory)
TRM_FILE = f"{REPO_DIR}/models/recursive_reasoning/trm.py"
!git -C {REPO_DIR} checkout -- models/recursive_reasoning/trm.py
with open(TRM_FILE, "r") as f:
    content = f.read()
old = """        # H_cycles-1 without grad
        with torch.no_grad():
            for _H_step in range(self.config.H_cycles-1):
                for _L_step in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
        # 1 with grad
        for _L_step in range(self.config.L_cycles):
            z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)"""
new = """        # 1-step gradient (O(1) memory)
        with torch.no_grad():
            for _H_step in range(self.config.H_cycles-1):
                for _L_step in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
            for _L_step in range(self.config.L_cycles):
                z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
            z_H = self.L_level(z_H, z_L, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)"""
assert content.replace(old, new) != content, "Patch not found!"
content = content.replace(old, new)
with open(TRM_FILE, "w") as f:
    f.write(content)
print("1-step gradient patch applied.")

1-step gradient patch applied.


## 4. Training Config

hidden=384, L_layers=2, 15000 epochs, eval every 5000

In [7]:
config = {
    "arch": {
        "name": "recursive_reasoning.trm@TinyRecursiveReasoningModel_ACTV1",
        "loss": {"name": "losses@ACTLossHead", "loss_type": "stablemax_cross_entropy"},
        "halt_exploration_prob": 0.1, "halt_max_steps": 4,
        "H_cycles": 3, "L_cycles": 6, "H_layers": 0, "L_layers": 2,
        "hidden_size": 384, "num_heads": 8, "expansion": 4,
        "puzzle_emb_ndim": 384, "pos_encodings": "rope",
        "forward_dtype": "float32", "mlp_t": False,
        "puzzle_emb_len": 16, "no_ACT_continue": True
    },
    "data_paths": ["data/sudoku-extreme-1k"],
    "data_paths_test": [], "evaluators": [],
    "global_batch_size": 512, "epochs": 10000, "eval_interval": 5000,
    "checkpoint_every_eval": True, "min_eval_interval": 0,
    "lr": 1e-4, "lr_min_ratio": 1.0, "lr_warmup_steps": 2000,
    "beta1": 0.9, "beta2": 0.95, "weight_decay": 0.1,
    "puzzle_emb_weight_decay": 0.1, "puzzle_emb_lr": 1e-2,
    "seed": 0, "ema": True, "ema_rate": 0.999, "freeze_weights": False,
    "project_name": "TRM-1step-Final", "run_name": "trm_1step_final",
    "checkpoint_path": "checkpoints/trm_1step_final"
}
os.makedirs("config/task", exist_ok=True)
with open("config/task/run.yaml", "w") as f:
    yaml.dump(config, f)
print("Config written.")

Config written.


In [8]:
%env WANDB_MODE=disabled
%env DISABLE_COMPILE=1
%env TQDM_POSITION=-1
print(f"Launching training with {GPU_COUNT} GPUs (DDP)...\n")
!torchrun --nproc_per_node={GPU_COUNT} pretrain.py --config-path config/task --config-name run

env: WANDB_MODE=disabled
env: DISABLE_COMPILE=1
env: TQDM_POSITION=-1
Launching training with 2 GPUs (DDP)...

W0528 12:26:39.682000 87 torch/distributed/run.py:852] 
W0528 12:26:39.682000 87 torch/distributed/run.py:852] *****************************************
W0528 12:26:39.682000 87 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0528 12:26:39.682000 87 torch/distributed/run.py:852] *****************************************
[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puz

## 5. Results

| | Dual L/H | Single flat |
|---|---|---|
| 1-step (O1) | HRM (55%) | **THIS RUN** |
| Full BP (OT) | TBD | TRM (87%) |

Checkpoints at epochs 5000, 10000, 15000. Run eval cell below.

In [9]:
# Load checkpoint and evaluate
import glob as _glob, json as _json, importlib
from tqdm import tqdm
from models.losses import ACTLossHead
from utils.functions import load_model_class
importlib.reload(__import__("models.recursive_reasoning.trm", fromlist=["*"]))
checkpoint_dir = "checkpoints/trm_1step_final"
ckpts = sorted(_glob.glob(f"{checkpoint_dir}/step_*"))
if not ckpts:
    print("No checkpoints found.")
else:
    print(f"Checkpoints: {len(ckpts)}")
    with open("data/sudoku-extreme-1k/test/dataset.json") as f:
        meta = _json.load(f)
    model_cfg = dict(batch_size=256, vocab_size=meta["vocab_size"], seq_len=meta["seq_len"], num_puzzle_identifiers=meta["num_puzzle_identifiers"], causal=False, halt_exploration_prob=0.1, halt_max_steps=4, H_cycles=3, L_cycles=6, H_layers=0, L_layers=2, hidden_size=384, num_heads=8, expansion=4, puzzle_emb_ndim=384, pos_encodings="rope", forward_dtype="float32", mlp_t=False, puzzle_emb_len=16, no_ACT_continue=True)
    model_cls = load_model_class("recursive_reasoning.trm@TinyRecursiveReasoningModel_ACTV1")
    with torch.device("cuda"):
        model = ACTLossHead(model_cls(model_cfg), "stablemax_cross_entropy")
    state = torch.load(ckpts[-1], map_location="cuda")
    model.load_state_dict(state, strict=False)
    model.cuda()
    print(f"Loaded: {os.path.basename(ckpts[-1])}")
    SUBSET = 2000
    total_correct, total_tokens, total_exact, total_examples = 0, 0, 0, 0
    model.eval()
    with torch.inference_mode():
        for split in meta["sets"]:
            labels = np.load(f"data/sudoku-extreme-1k/test/{split}__labels.npy", mmap_mode="r")
            inputs_np = np.load(f"data/sudoku-extreme-1k/test/{split}__inputs.npy", mmap_mode="r")
            pids_np = np.load(f"data/sudoku-extreme-1k/test/{split}__puzzle_identifiers.npy", mmap_mode="r")
            n_eval = min(len(labels), SUBSET)
            pbar = tqdm(total=n_eval, desc="Eval", unit="puzzles", position=0, leave=True)
            for i in range(0, n_eval, 128):
                end = min(i + 128, n_eval)
                with torch.device("cuda"):
                    batch = {"inputs": torch.tensor(inputs_np[i:end]), "labels": torch.tensor(labels[i:end]), "puzzle_identifiers": torch.tensor(pids_np[i:end])}
                    carry = model.initial_carry(batch)
                while True:
                    carry, _, _, logits_dict, all_finish = model(carry=carry, batch=batch, return_keys=["logits"])
                    if all_finish:
                        break
                logits = logits_dict["logits"]
                mask = batch["labels"] != 0
                pred_tokens = torch.argmax(logits, dim=-1)
                total_correct += ((pred_tokens == batch["labels"]) & mask).sum().item()
                total_tokens += mask.sum().item()
                total_exact += ((pred_tokens == batch["labels"]) | ~mask).all(dim=-1).sum().item()
                total_examples += (end - i)
                pbar.update(end - i)
            pbar.close()
    print(f"\n=== RESULTS ===\nToken accuracy:  {total_correct / total_tokens * 100:.1f}%\nExact accuracy:  {total_exact / total_examples * 100:.1f}%\nTest examples:   {total_examples}")


Checkpoints: 2
Loaded: step_9764


Eval: 100%|██████████| 2000/2000 [00:49<00:00, 40.42puzzles/s]


=== RESULTS ===
Token accuracy:  65.9%
Exact accuracy:  2.2%
Test examples:   2000
